### Q10 Which brain area has the shortest intrinsic neuronal timescale during baseline activity?

Intrinsic neuronal timescales are estimated for every single unit using three trial-based methods from `isttc`: Pearson trial-average, STTC trial-average, and STTC trial-concat. Each method uses the same 250 one-second baseline intervals for every unit.

Baseline activity was defined using 1-s intervals immediately preceding trial onset, separated from `trial_start` by a 50-ms guard period; thus, each candidate interval spanned `[trial_start - 1.05 s, trial_start - 0.05 s)`. A single continuous baseline period was not used because the longest trial-free segment varied substantially across mice and was only 9.7 s for one mouse, making it too short for a reliable duration-matched analysis. Candidate intervals were excluded if they overlapped the preceding trial, or could not be verified because the preceding `trial_end` was missing. Each mouse had at least 256 valid intervals. To ensure equal signal duration across mice, 250 intervals were sampled without replacement for each mouse using a fixed random seed (`42`), yielding 4,500 intervals in total.

Spike timestamps are stored in seconds but the `isttc` functions operate on integer time points. Within every baseline interval, spikes are realigned to interval onset and converted to 30-kHz sample indices before calculating autocorrelation. The exponential fit returns the time constant in 50-ms lag steps; fitted values and confidence bounds are therefore multiplied by 50 to obtain milliseconds.

In [ ]:
import csv
import io
import os
import pickle
import sys
import tempfile
import time
from contextlib import redirect_stdout
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

#### Parameters

In [ ]:
project_folder = Path("E:/con2phys_submission")
dataset_folder = project_folder / "dataset"
sua_csv_path = dataset_folder / "full_sua_dataset.csv"
baseline_csv_path = dataset_folder / "baseline_periods.csv"
isttc_repo_path = Path("D:/intr_timescales/isttc")
q_path = Path(os.environ.get(
    "Q10_OUTPUT_DIR", "D:/git_repos/con2phys_submission/results/q10"
))
results_csv_path = q_path / "sua_it_methods.csv"
acf_pickle_path = q_path / "sua_it_acfs.pkl"
checkpoint_path = q_path / "sua_it_checkpoint.pkl"
run_log_path = q_path / "sua_it_run_times.csv"
figure_path = q_path / "sua_it_methods.png"

SAMPLING_RATE_HZ = 30_000
TRIAL_LENGTH_SEC = 1.0
BIN_SIZE_MS = 50
STTC_DT_CONCAT_MS = 25
STTC_AVG_PADDING_MS = 150
STTC_CONCAT_PADDING_MS = 3_000
EXPECTED_N_MICE = 18
EXPECTED_N_UNITS = 1_449
N_INTERVALS_PER_MOUSE = 250
METHODS = ("pearsonr_trial_avg", "sttc_trial_avg", "sttc_trial_concat")
CHECKPOINT_EVERY = 25
MAX_UNITS = int(os.environ["Q10_MAX_UNITS"]) if os.environ.get("Q10_MAX_UNITS") else None

TRIAL_LENGTH_SAMPLES = int(TRIAL_LENGTH_SEC * SAMPLING_RATE_HZ)
BIN_SIZE_SAMPLES = int(BIN_SIZE_MS * SAMPLING_RATE_HZ / 1_000)
N_LAGS = TRIAL_LENGTH_SAMPLES // BIN_SIZE_SAMPLES
STTC_DT_AVG_SAMPLES = BIN_SIZE_SAMPLES - 1
STTC_DT_CONCAT_SAMPLES = int(STTC_DT_CONCAT_MS * SAMPLING_RATE_HZ / 1_000)
STTC_AVG_PADDING_SAMPLES = int(STTC_AVG_PADDING_MS * SAMPLING_RATE_HZ / 1_000)
STTC_CONCAT_PADDING_SAMPLES = int(STTC_CONCAT_PADDING_MS * SAMPLING_RATE_HZ / 1_000)

if str(isttc_repo_path) not in sys.path:
    sys.path.insert(0, str(isttc_repo_path))

from src.isttc.scripts.calculate_acf import (
    acf_pearsonr_trial_avg, acf_sttc_trial_avg, acf_sttc_trial_concat,
)
from src.isttc.scripts.calculate_tau import fit_single_exp, func_single_exp
from src.isttc.scripts.spike_train_utils import bin_trials

RUN_CONFIG = {
    "sampling_rate_hz": SAMPLING_RATE_HZ,
    "trial_length_samples": TRIAL_LENGTH_SAMPLES,
    "bin_size_samples": BIN_SIZE_SAMPLES,
    "n_lags": N_LAGS,
    "sttc_dt_avg_samples": STTC_DT_AVG_SAMPLES,
    "sttc_dt_concat_samples": STTC_DT_CONCAT_SAMPLES,
    "sttc_avg_padding_samples": STTC_AVG_PADDING_SAMPLES,
    "sttc_concat_padding_samples": STTC_CONCAT_PADDING_SAMPLES,
    "n_intervals_per_mouse": N_INTERVALS_PER_MOUSE,
    "methods": METHODS,
    "max_units": MAX_UNITS,
    "qc_definition": "r2_ge_0.5_ci_excludes_zero_acf_decline_50_200ms",
}
RUN_CONFIG

#### Load and validate baseline intervals

In [ ]:
required_paths = [sua_csv_path, baseline_csv_path, isttc_repo_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Required paths not found: {missing_paths}")

baseline_df = pd.read_csv(baseline_csv_path)
expected_baseline_columns = ["mouse_id", "trial_id", "baseline_start", "baseline_end"]
if list(baseline_df.columns) != expected_baseline_columns:
    raise ValueError(f"Unexpected baseline schema: {list(baseline_df.columns)}")
if baseline_df.isna().any().any():
    raise ValueError("Baseline table contains missing values")
if baseline_df[["mouse_id", "trial_id"]].duplicated().any():
    raise ValueError("Duplicate selected trials found in the baseline table")

interval_counts = baseline_df.groupby("mouse_id").size()
if len(interval_counts) != EXPECTED_N_MICE:
    raise ValueError(f"Expected {EXPECTED_N_MICE} mice, found {len(interval_counts)}")
if not interval_counts.eq(N_INTERVALS_PER_MOUSE).all():
    raise ValueError("Every mouse must have exactly 250 baseline intervals")
interval_durations = baseline_df["baseline_end"] - baseline_df["baseline_start"]
if not np.allclose(interval_durations, TRIAL_LENGTH_SEC, rtol=0, atol=1e-9):
    raise ValueError("Every baseline interval must be exactly one second")

baseline_intervals_by_mouse = {
    int(mouse_id): group.sort_values("baseline_start")[
        ["baseline_start", "baseline_end"]
    ].to_numpy(dtype=float)
    for mouse_id, group in baseline_df.groupby("mouse_id")
}
display(interval_counts.rename("n_intervals").reset_index())
print(f"Validated {len(baseline_df):,} one-second baseline intervals.")

#### Calculation helpers

The approximately 802 MB SUA CSV is streamed one unit at a time. A checkpoint is saved every 25 completed units. Per-unit method time is stored in the results table, and every notebook run is appended to a run-level timing log.

In [ ]:
def atomic_pickle_dump(payload, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="wb", dir=destination.parent, prefix=f"{destination.stem}_",
            suffix=".tmp", delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
        temporary_path.replace(destination)
    except Exception:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)
        raise


def atomic_csv_write(dataframe: pd.DataFrame, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".tmp")
    try:
        dataframe.to_csv(temporary_path, index=False)
        temporary_path.replace(destination)
    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise


def extract_baseline_trials(spike_times_sec, intervals_sec):
    trials = []
    for interval_start, interval_end in intervals_sec:
        left = np.searchsorted(spike_times_sec, interval_start, side="left")
        right = np.searchsorted(spike_times_sec, interval_end, side="left")
        relative_samples = (
            (spike_times_sec[left:right] - interval_start) * SAMPLING_RATE_HZ
        ).astype(np.int64)
        relative_samples = relative_samples[
            (relative_samples >= 0) & (relative_samples < TRIAL_LENGTH_SAMPLES)
        ]
        trials.append(relative_samples)
    return trials


def fit_acf(acf_values):
    with redirect_stdout(io.StringIO()):
        fit = fit_single_exp(acf_values, start_idx_=1, exp_fun_=func_single_exp)
    popt, pcov, tau_steps, tau_ci, fit_r_squared, explained_var, status = fit
    popt_array = np.asarray(popt) if not np.isscalar(popt) else np.asarray([])
    return {
        "tau": tau_steps,
        "tau_steps": tau_steps,
        "tau_lower": tau_ci[0],
        "tau_upper": tau_ci[1],
        "tau_ms": tau_steps * BIN_SIZE_MS,
        "tau_lower_ms": tau_ci[0] * BIN_SIZE_MS,
        "tau_upper_ms": tau_ci[1] * BIN_SIZE_MS,
        "fit_r_squared": fit_r_squared,
        "explained_var": explained_var,
        "fit_a": popt_array[0] if popt_array.size == 3 else np.nan,
        "fit_c": popt_array[2] if popt_array.size == 3 else np.nan,
        "fit_status": status,
        "log_message": status,
        "fit_pcov": pcov,
    }


def calculate_acf_decline(acf_values, start_ms=50, end_ms=200):
    lag_ms = np.arange(len(acf_values)) * BIN_SIZE_MS
    window = np.asarray(acf_values, dtype=float)[
        (lag_ms >= start_ms) & (lag_ms <= end_ms)
    ]
    return bool(
        window.size >= 2
        and np.all(np.isfinite(window))
        and np.all(np.diff(window) <= 0)
    )


def calculate_common_qc(acf_values, fit_result):
    tau_lower_ms = fit_result["tau_lower_ms"]
    tau_upper_ms = fit_result["tau_upper_ms"]
    ci_zero_excluded = int(tau_lower_ms > 0 or tau_upper_ms < 0)
    acf_decline = calculate_acf_decline(acf_values)
    qc_pass = bool(
        fit_result["fit_r_squared"] >= 0.5
        and ci_zero_excluded == 1
        and acf_decline
    )
    return {
        "ci_zero_excluded": ci_zero_excluded,
        "ci_width_ms": tau_upper_ms - tau_lower_ms,
        "acf_decline": acf_decline,
        "qc_pass": qc_pass,
    }


def calculate_unit_methods(metadata, spike_times_sec):
    mouse_id = metadata["mouse_id"]
    trials = extract_baseline_trials(spike_times_sec, baseline_intervals_by_mouse[mouse_id])
    if len(trials) != N_INTERVALS_PER_MOUSE:
        raise ValueError(f"Unit {metadata['unit_key']} has {len(trials)} trials")
    trial_counts = np.asarray([len(trial) for trial in trials])
    n_baseline_spikes = int(trial_counts.sum())
    unit_summary = {
        **metadata,
        "n_baseline_intervals": len(trials),
        "baseline_duration_sec": float(len(trials) * TRIAL_LENGTH_SEC),
        "n_baseline_spikes": n_baseline_spikes,
        "baseline_firing_rate_hz": n_baseline_spikes / (len(trials) * TRIAL_LENGTH_SEC),
        "n_nonempty_intervals": int((trial_counts > 0).sum()),
    }
    binned_trials = bin_trials(trials, TRIAL_LENGTH_SAMPLES, BIN_SIZE_SAMPLES)
    if binned_trials.shape != (N_INTERVALS_PER_MOUSE, N_LAGS):
        raise ValueError(f"Unexpected binned shape: {binned_trials.shape}")

    calculations = (
        ("pearsonr_trial_avg", lambda: acf_pearsonr_trial_avg(
            binned_trials, N_LAGS, verbose_=False)),
        ("sttc_trial_avg", lambda: acf_sttc_trial_avg(
            trials, N_LAGS, BIN_SIZE_SAMPLES, STTC_DT_AVG_SAMPLES,
            STTC_AVG_PADDING_SAMPLES, verbose_=False)),
        ("sttc_trial_concat", lambda: acf_sttc_trial_concat(
            trials, N_LAGS, BIN_SIZE_SAMPLES, STTC_DT_CONCAT_SAMPLES,
            TRIAL_LENGTH_SAMPLES, STTC_CONCAT_PADDING_SAMPLES, verbose_=False)),
    )
    result_rows, unit_acfs = [], {}
    for method, calculation in calculations:
        started = time.perf_counter()
        acf_matrix = None
        try:
            with redirect_stdout(io.StringIO()):
                method_output = calculation()
            if method == "sttc_trial_concat":
                acf_values = np.asarray(method_output, dtype=float)
            else:
                acf_matrix, acf_values = method_output
                acf_matrix = np.asarray(acf_matrix, dtype=float)
                acf_values = np.asarray(acf_values, dtype=float)
            fit_result = fit_acf(acf_values)
            elapsed = time.perf_counter() - started
            result_rows.append({
                **unit_summary, "method": method,
                "n_finite_acf_lags": int(np.isfinite(acf_values).sum()),
                "elapsed_seconds": elapsed,
                **{k: v for k, v in fit_result.items() if k != "fit_pcov"},
                **calculate_common_qc(acf_values, fit_result),
            })
            unit_acfs[method] = {
                "lag_ms": np.arange(acf_values.size) * BIN_SIZE_MS,
                "acf": acf_values, "acf_matrix": acf_matrix,
                "fit_pcov": fit_result["fit_pcov"],
            }
        except Exception as error:
            elapsed = time.perf_counter() - started
            result_rows.append({
                **unit_summary, "method": method, "n_finite_acf_lags": 0,
                "elapsed_seconds": elapsed, "tau": np.nan,
                "tau_steps": np.nan, "tau_lower": np.nan,
                "tau_upper": np.nan, "tau_ms": np.nan,
                "tau_lower_ms": np.nan,
                "tau_upper_ms": np.nan, "fit_r_squared": np.nan,
                "explained_var": np.nan, "fit_a": np.nan, "fit_c": np.nan,
                "fit_status": f"{type(error).__name__}: {error}",
                "log_message": f"{type(error).__name__}: {error}",
                "ci_zero_excluded": 0, "ci_width_ms": np.nan,
                "acf_decline": False, "qc_pass": False,
            })
            unit_acfs[method] = {
                "lag_ms": np.arange(N_LAGS) * BIN_SIZE_MS,
                "acf": np.full(N_LAGS, np.nan), "acf_matrix": None,
                "fit_pcov": np.nan,
            }
    return result_rows, unit_acfs


def save_checkpoint(result_rows, acf_results, processed_units):
    atomic_pickle_dump({
        "config": RUN_CONFIG, "result_rows": result_rows,
        "acf_results": acf_results,
        "processed_units": sorted(processed_units),
    }, checkpoint_path)

#### Calculate single-unit timescales

All units are processed unless `MAX_UNITS` is set for a pilot. A compatible checkpoint is resumed automatically. The results contain method-level elapsed time, while `sua_it_run_times.csv` records wall-clock and cumulative method time for every execution.

In [ ]:
q_path.mkdir(parents=True, exist_ok=True)
run_started_at = datetime.now()
run_started = time.perf_counter()
run_method_seconds = {method: 0.0 for method in METHODS}

if checkpoint_path.exists():
    with checkpoint_path.open("rb") as handle:
        checkpoint = pickle.load(handle)
    if checkpoint.get("config") != RUN_CONFIG:
        raise ValueError("Checkpoint parameters differ from RUN_CONFIG")
    result_rows = checkpoint["result_rows"]
    acf_results = checkpoint["acf_results"]
    processed_units = {tuple(unit) for unit in checkpoint["processed_units"]}
    print(f"Resuming from {len(processed_units):,} completed units.")
else:
    result_rows, acf_results, processed_units = [], {}, set()

csv.field_size_limit(2**31 - 1)
newly_processed = 0
with sua_csv_path.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    expected_sua_columns = [
        "mouse_id", "cluster_id", "brain_area", "recording_duration_sec",
        "unit_last_spike_sec", "n_spikes", "spikes",
    ]
    if reader.fieldnames != expected_sua_columns:
        raise ValueError(f"Unexpected SUA schema: {reader.fieldnames}")
    for source_index, row in enumerate(reader):
        if MAX_UNITS is not None and source_index >= MAX_UNITS:
            break
        mouse_id, cluster_id = int(row["mouse_id"]), int(row["cluster_id"])
        unit_key = (mouse_id, cluster_id)
        if unit_key in processed_units:
            continue
        spike_times_sec = np.fromstring(row["spikes"], sep=";")
        expected_n_spikes = int(row["n_spikes"])
        if spike_times_sec.size != expected_n_spikes:
            raise ValueError(f"Spike-count mismatch for {unit_key}")
        if not np.isfinite(spike_times_sec).all() or np.any(np.diff(spike_times_sec) < 0):
            raise ValueError(f"Invalid spike times for {unit_key}")
        metadata = {
            "source_index": source_index, "unit_key": f"{mouse_id}:{cluster_id}",
            "mouse_id": mouse_id, "cluster_id": cluster_id,
            "brain_area": int(row["brain_area"]),
            "recording_duration_sec": float(row["recording_duration_sec"]),
            "unit_total_spikes": expected_n_spikes,
        }
        unit_rows, unit_acfs = calculate_unit_methods(metadata, spike_times_sec)
        result_rows.extend(unit_rows)
        acf_results[unit_key] = unit_acfs
        processed_units.add(unit_key)
        newly_processed += 1
        for result in unit_rows:
            run_method_seconds[result["method"]] += result["elapsed_seconds"]
        if len(processed_units) % CHECKPOINT_EVERY == 0:
            save_checkpoint(result_rows, acf_results, processed_units)
            print(
                f"{datetime.now():%Y-%m-%d %H:%M:%S}: completed "
                f"{len(processed_units):,} units in "
                f"{(time.perf_counter() - run_started) / 60:.1f} min this run"
            )

if newly_processed:
    save_checkpoint(result_rows, acf_results, processed_units)
expected_units = MAX_UNITS if MAX_UNITS is not None else EXPECTED_N_UNITS
if len(processed_units) != expected_units:
    raise ValueError(f"Expected {expected_units:,} units, found {len(processed_units):,}")

results_df = pd.DataFrame(result_rows).sort_values(["source_index", "method"]).reset_index(drop=True)
if len(results_df) != expected_units * len(METHODS):
    raise ValueError("Unexpected result-row count")
if not results_df.groupby(["mouse_id", "cluster_id"])["method"].nunique().eq(len(METHODS)).all():
    raise ValueError("Every unit must have exactly one result per method")

atomic_csv_write(results_df, results_csv_path)
atomic_pickle_dump({"config": RUN_CONFIG, "acf_results": acf_results}, acf_pickle_path)
checkpoint_path.unlink(missing_ok=True)

run_finished_at = datetime.now()
wall_time_seconds = time.perf_counter() - run_started
run_record = {
    "started_at": run_started_at.isoformat(timespec="seconds"),
    "finished_at": run_finished_at.isoformat(timespec="seconds"),
    "wall_time_seconds": wall_time_seconds,
    "new_units_processed": newly_processed,
    "total_units_in_results": len(processed_units),
    "result_rows": len(results_df),
    "max_units_setting": MAX_UNITS,
    **{f"{method}_seconds": seconds for method, seconds in run_method_seconds.items()},
}
if run_log_path.exists():
    run_log_df = pd.concat([pd.read_csv(run_log_path), pd.DataFrame([run_record])], ignore_index=True)
else:
    run_log_df = pd.DataFrame([run_record])
atomic_csv_write(run_log_df, run_log_path)

print(
    f"Saved {len(results_df):,} results for {len(processed_units):,} units in "
    f"{wall_time_seconds / 60:.1f} minutes this run."
)
display(pd.DataFrame([run_record]))
results_df.head()

#### Fit diagnostics and descriptive plot

Failed fits are retained with missing timescales and their diagnostic status. `qc_pass` matches the common SUA quality control: fit R-squared at least 0.5, a tau confidence interval excluding zero, and a monotonically non-increasing ACF from 50 through 200 ms. The separate 50--1000 ms tau-range restriction is intentionally left for the downstream analysis. The plot shows QC-passing units and is descriptive; it does not account for multiple units recorded from the same mouse.

In [ ]:
fit_summary = (
    results_df.assign(fit_ok=results_df["fit_status"].eq("ok"))
    .groupby(["method", "brain_area"])
    .agg(
        n_units=("cluster_id", "size"), n_fit_ok=("fit_ok", "sum"),
        n_qc_pass=("qc_pass", "sum"),
        median_tau_ms=("tau_ms", "median"),
        median_r_squared=("fit_r_squared", "median"),
        median_baseline_rate_hz=("baseline_firing_rate_hz", "median"),
    ).reset_index()
)
display(fit_summary)
display(results_df["fit_status"].value_counts(dropna=False))

plot_df = results_df.loc[
    results_df["qc_pass"] & np.isfinite(results_df["tau_ms"])
    & results_df["tau_ms"].gt(0)
].copy()
fig, axes = plt.subplots(1, len(METHODS), figsize=(12, 4), sharey=True)
for axis, method in zip(axes, METHODS):
    method_df = plot_df.loc[plot_df["method"].eq(method)]
    sns.boxplot(data=method_df, x="brain_area", y="tau_ms", showfliers=False,
                color="lightsteelblue", ax=axis)
    sns.stripplot(data=method_df, x="brain_area", y="tau_ms", color="black",
                  alpha=0.25, size=2, ax=axis)
    axis.set_yscale("log")
    axis.set_title(method.replace("_", " ").title())
    axis.set_xlabel("Brain area")
    axis.set_ylabel("Timescale (ms)" if axis is axes[0] else "")
    sns.despine(ax=axis)
fig.suptitle("Single-unit intrinsic timescales during baseline activity")
fig.tight_layout()
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Match the common SUA selection used in the timescales project:
# model-level qc_pass, followed by the separate inclusive 50--1000 ms range.
sttc_concat_qc = results_df.loc[
    results_df["method"].eq("sttc_trial_concat")
    & results_df["qc_pass"].eq(True)
    & results_df["tau_ms"].between(50, 1_000, inclusive="both")
].copy()

sttc_concat_summary = (
    sttc_concat_qc.groupby("brain_area")["tau_ms"]
    .agg(mean_tau_ms="mean", sd_tau_ms="std", n_units="count")
)
sttc_concat_summary["sem_tau_ms"] = (
    sttc_concat_summary["sd_tau_ms"]
    / np.sqrt(sttc_concat_summary["n_units"])
)
sttc_concat_summary["ci95_margin_ms"] = (
    1.96 * sttc_concat_summary["sem_tau_ms"]
)
sttc_concat_summary["ci95_low_ms"] = (
    sttc_concat_summary["mean_tau_ms"]
    - sttc_concat_summary["ci95_margin_ms"]
)
sttc_concat_summary["ci95_high_ms"] = (
    sttc_concat_summary["mean_tau_ms"]
    + sttc_concat_summary["ci95_margin_ms"]
)
summary_path = q_path / "sttc_concat_tau_mean_and_95_ci_qc_50_1000ms.csv"
sttc_concat_summary.to_csv(summary_path)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
plt.subplots_adjust(wspace=0.45)

sns.violinplot(
    ax=axes[0], x="brain_area", y="tau_ms",
    data=sttc_concat_qc, cut=0, density_norm="width",
    color="lightsteelblue",
)
axes[0].set_xlabel("Brain area")
axes[0].set_ylabel("Timescale (ms)")
axes[0].grid(True, linestyle="-")

areas = sttc_concat_summary.index.to_numpy()
means = sttc_concat_summary["mean_tau_ms"].to_numpy()
ci95 = sttc_concat_summary["ci95_margin_ms"].to_numpy()
axes[1].errorbar(
    areas, means, yerr=ci95,
    fmt="o", color="black", markersize=6,
    capsize=4, linewidth=1.5,
)
axes[1].set_xticks(areas)
axes[1].set_xlabel("Brain area")
axes[1].set_ylabel("Mean timescale (ms)")
axes[1].set_title("Mean and 95% CI")
axes[1].grid(True, linestyle="-")

sns.despine()
figure_path = q_path / "sttc_concat_tau_mean_and_95_ci_qc_50_1000ms.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")

print(f"Selected {len(sttc_concat_qc):,} units.")
display(sttc_concat_summary)
plt.show()